[CAPSTONE INTEGRATION TARGET]
    Integrate validated dimensions into a projector mounting bracket and an open-top sensor housing. This notebook creates a configurable fixture candidate; exact fit, retention, optical alignment and load suitability require measured hardware and physical acceptance.


# 1. Declare the measurement basis
    Plato Shadows is an original project concept by Tee Ken Ng, @TeeKenNG on TikTok. This curriculum is an independent teaching implementation.

    The default JSON uses example millimetre values and does not describe a particular projector or webcam. To load your reviewed complete numeric record, set `CORTEXCAD_MEASUREMENTS` to its absolute path. The fixture assumes a rectangular four-hole base pattern, two centered upright holes of the same diameter, and centered front/rear housing apertures. If your hardware differs, adapt the recipe and tests before live use.


In [ ]:
from __future__ import annotations
import importlib.util
import json
import math
import os
from pathlib import Path
import sys

if sys.version_info < (3, 12):
    raise RuntimeError("Choose the CortexCAD Python 3.12+ Jupyter kernel")

# Installed packages work from any directory. A source checkout can be selected
# explicitly; ancestor discovery also supports launching Jupyter inside a phase.
if importlib.util.find_spec("cortexcad") is None:
    root_hint: str = os.environ.get("CORTEXCAD_ROOT", "")
    candidates: list[Path] = ([Path(root_hint).expanduser()] if root_hint else [])
    candidates += [Path.cwd(), *Path.cwd().parents]
    source_root: Path | None = next((path for path in candidates
                                    if (path / "cortexcad" / "recipes.py").is_file()), None)
    if source_root is None:
        raise RuntimeError("Install CortexCAD in this kernel or set CORTEXCAD_ROOT to its checkout")
    sys.path.insert(0, str(source_root.resolve()))

from cortexcad.client import FreeCADClient
from cortexcad.recipes import DEFAULTS, build_script, validate_parameters
LIVE: bool = os.environ.get("CORTEXCAD_LIVE") == "1"
print("LIVE execution enabled" if LIVE else "OFFLINE: no CAD connection will be opened")


In [ ]:
def read_numeric_object(text: str) -> dict[str, object]:
    """Reject ambiguous duplicate JSON keys before phase-specific validation."""
    def unique_pairs(pairs: list[tuple[str, object]]) -> dict[str, object]:
        result: dict[str, object] = {}
        for key, value in pairs:
            if key in result:
                raise ValueError(f"Duplicate JSON key: {key}")
            result[key] = value
        return result
    def reject_constant(value: str) -> object:
        raise ValueError(f"Non-finite JSON constant: {value}")
    value: object = json.loads(text, object_pairs_hook=unique_pairs, parse_constant=reject_constant)
    if not isinstance(value, dict):
        raise ValueError("Measurements must be a JSON object")
    return value


In [ ]:
PHASE: str = "capstone"
measurement_path: str = os.environ.get("CORTEXCAD_MEASUREMENTS", "")
if measurement_path:
    measurement_file: Path = Path(measurement_path).expanduser().resolve(strict=True)
    measurements: dict[str, object] = read_numeric_object(measurement_file.read_text(encoding="utf-8"))
    print("Loaded candidate measured record:", measurement_file.name)
else:
    measurements = read_numeric_object(json.dumps(DEFAULTS[PHASE], allow_nan=False))
    print("EXAMPLE FIXTURE ONLY: no hardware compatibility has been established")
if set(measurements) != set(DEFAULTS[PHASE]):
    raise ValueError("Capstone measurement records must include every documented key")
params: dict[str, float] = validate_parameters(PHASE, measurements)
print(json.dumps(params, indent=2, sort_keys=True))


## 2. Predict material and clearances
    The bracket joins a plate and upright, then removes four plate holes and two upright holes. The housing subtracts an open-top rectangular cavity and two circular wall apertures. It is displayed separately from the bracket; no mechanical attachment between them is implied. A retaining lid, strain relief, projector-specific hole layout, and assembled positioning require measured extensions when applicable.


In [ ]:
inner_length: float = params["sensor_length"] + 2 * params["clearance"]
inner_width: float = params["sensor_width"] + 2 * params["clearance"]
inner_height: float = params["sensor_height"] + params["clearance"]
wall: float = params["wall"]
expected_mount: float = (params["plate_length"] * params["plate_width"] * params["plate_thickness"]
    + params["plate_length"] * params["plate_thickness"] * params["bracket_height"]
    - 6 * math.pi * (params["fastener_diameter"] / 2) ** 2 * params["plate_thickness"])
expected_housing: float = ((inner_length + 2 * wall) * (inner_width + 2 * wall) * (inner_height + wall)
    - inner_length * inner_width * inner_height
    - math.pi * ((params["lens_diameter"] / 2) ** 2 + (params["cable_diameter"] / 2) ** 2) * wall)
assert expected_mount > 0 and expected_housing > 0
print("Cavity dimensions (mm):", (inner_length, inner_width, inner_height))
print("Expected candidate volumes (mm³):", expected_mount, expected_housing)
print("Physical fit verified:", False)


In [ ]:
script: str = build_script(PHASE, params)
compile(script, "freecad_lesson.py", "exec")
print(script)
print("Compiled source only: no FreeCAD solver or solid has run on this host.")


In [ ]:
if LIVE:
    # Endpoint and explicit timeout come from CORTEXCAD_ENDPOINT/TIMEOUT.
    client: FreeCADClient = FreeCADClient(timeout=float(os.environ.get("CORTEXCAD_TIMEOUT", "5")))
    outcome = client.run_recipe(PHASE, params)
    print(repr(outcome))
    print("Inspect the workstation and native assertions before accepting this lesson.")
else:
    print("OFFLINE preview complete; native geometry and hardware-fit gates remain pending.")


## 3. Falsify impossible layouts
    These checks reject mounting holes that consume edge material, impossible aperture diameters, and excessive clearance. Bounds constrain the supported exercise; they do not prove strength or manufacturing capability.


In [ ]:
for invalid in ({"mount_pitch_x": params["plate_length"]},
                {"mount_pitch_y": params["fastener_diameter"]},
                {"lens_diameter": max(inner_length, inner_height) + 1},
                {"clearance": 11}):
    try:
        build_script(PHASE, {**params, **invalid})
    except ValueError as error:
        print("Expected rejection:", error)
    else:
        raise AssertionError("Invalid fixture accepted")


## 4. Acceptance requires the complete installation
    Follow README gates M0–M6: measurement provenance, model suitability, real CAD, fit coupon, retained hardware and load review, optical/cable/thermal access, then repeatable projector/webcam integration. Record every result as pending/pass/fail with evidence. CAD validity and analytic volume are necessary but cannot establish fit or strength.

    Provide Build environment with source and measured revisions, and DevOps with the signed integration record. They own version-control transition and deployment. The initial handover has no measured dimensions or workstation evidence, so exact fit and release approval remain open.
